# Slicing CT-images to make them available to Dino

In [1]:
import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import torch.nn.functional as F
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
def resize_array(array, current_spacing, target_spacing):
    """
    Resize the array to match the target spacing.

    Args:
    array (torch.Tensor): Input array to be resized.
    current_spacing (tuple): Current voxel spacing (z_spacing, xy_spacing, xy_spacing).
    target_spacing (tuple): Target voxel spacing (target_z_spacing, target_x_spacing, target_y_spacing).

    Returns:
    np.ndarray: Resized array.
    """
    # Calculate new dimensions
    original_shape = array.shape[2:]
    scaling_factors = [
        current_spacing[i] / target_spacing[i] for i in range(len(original_shape))
    ]
    new_shape = [
        int(original_shape[i] * scaling_factors[i]) for i in range(len(original_shape))
    ]
    # Resize the array
    resized_array = F.interpolate(array, size=new_shape, mode='trilinear', align_corners=False).cpu().numpy()
    return resized_array

def nii_img_to_tensor(path):
    nii_img = nib.load(str(path))
    img_data = nii_img.get_fdata()

    slope = 1.0
    intercept = 0.0
    pixdim = nii_img.header.get_zooms()  # (x, y, z)
    xy_spacing = float(pixdim[0])
    z_spacing = float(pixdim[2])
    # print(f"Original spacing: x={xy_spacing}, y={xy_spacing}, z={z_spacing}")
    # Define the target spacing values
    target_x_spacing = 0.75
    target_y_spacing = 0.75
    target_z_spacing = 1.5
    # print(f"Target spacing: x={target_x_spacing}, y={target_y_spacing}, z={target_z_spacing}")

    current = (z_spacing, xy_spacing, xy_spacing)
    target = (target_z_spacing, target_x_spacing, target_y_spacing)

    img_data = slope * img_data + intercept
    hu_min, hu_max = -1000, 1000
    img_data = np.clip(img_data, hu_min, hu_max)

    img_data = img_data.transpose(2, 0, 1)

    tensor = torch.tensor(img_data)
    tensor = tensor.unsqueeze(0).unsqueeze(0)

    img_data = resize_array(tensor, current, target)
    img_data = img_data[0][0]
    img_data= np.transpose(img_data, (1, 2, 0))

    img_data = (((img_data ) / 1000)).astype(np.float32)
    slices=[]

    tensor = torch.tensor(img_data)
    # Get the dimensions of the input tensor
    target_shape = (480,480,240)

    # Extract dimensions
    h, w, d = tensor.shape
    # print(f"Original shape: h={h}, w={w}, d={d}")
    # print(f"Target shape: h={target_shape[0]}, w={target_shape[1]}, d={target_shape[2]}")

    # Calculate cropping/padding values for height, width, and depth
    dh, dw, dd = target_shape
    h_start = max((h - dh) // 2, 0)
    h_end = min(h_start + dh, h)
    w_start = max((w - dw) // 2, 0)
    w_end = min(w_start + dw, w)
    d_start = max((d - dd) // 2, 0)
    d_end = min(d_start + dd, d)

    # Crop or pad the tensor
    tensor = tensor[h_start:h_end, w_start:w_end, d_start:d_end]

    pad_h_before = (dh - tensor.size(0)) // 2
    pad_h_after = dh - tensor.size(0) - pad_h_before

    pad_w_before = (dw - tensor.size(1)) // 2
    pad_w_after = dw - tensor.size(1) - pad_w_before

    pad_d_before = (dd - tensor.size(2)) // 2
    pad_d_after = dd - tensor.size(2) - pad_d_before

    tensor = torch.nn.functional.pad(tensor, (pad_d_before, pad_d_after, pad_w_before, pad_w_after, pad_h_before, pad_h_after), value=-1)

    tensor = tensor.permute(2, 0, 1)

    tensor = tensor.unsqueeze(0)
    if tensor.ndim == 4:
        tensor = tensor.unsqueeze(1)

    return tensor

def plot_ct_slice(ct_tensor: torch.Tensor, slice_index: int):
    if ct_tensor.ndim != 5 or ct_tensor.shape[0] != 1 or ct_tensor.shape[1] != 1:
        raise ValueError(f"Expected shape [1, 1, D, H, W], got {tuple(ct_tensor.shape)}")
    
    depth = ct_tensor.shape[2]
    if not (0 <= slice_index < depth):
        raise ValueError(f"slice_index must be between 0 and {depth-1}, got {slice_index}")
    
    slice_2d = ct_tensor[0, 0, slice_index, :, :].cpu().numpy()
    
    plt.figure(figsize=(6, 6))
    plt.imshow(slice_2d, cmap='gray')
    plt.title(f"CT Slice {slice_index}/{depth-1}")
    plt.axis('off')
    plt.show()

def slice_ct(ct_tensor: torch.Tensor, slice_index: int):
    if ct_tensor.ndim != 5 or ct_tensor.shape[0] != 1 or ct_tensor.shape[1] != 1:
        raise ValueError(f"Expected shape [1, 1, D, H, W], got {tuple(ct_tensor.shape)}")
    
    depth = ct_tensor.shape[2]
    if not (0 <= slice_index < depth):
        raise ValueError(f"slice_index must be between 0 and {depth-1}, got {slice_index}")
    
    return ct_tensor[0, 0, slice_index, :, :].cpu().numpy()